# TT/MPS 圧縮における ReLU の誤差伝播
## 1-Lipschitz 性と Frobenius norm による hidden activation error の評価

配置想定：

```text
notebooks/
└── 30_tt_mps/
    └── 10_fashion_mnist_mlp/
        ├── 00_mlp_ttlinear_logits_equivalence.ipynb
        ├── 01_ttlinear_output_error_norm_bounds.ipynb
        └── 02_relu_hidden_activation_error.ipynb
```

## 今回の範囲

今回は、

$$
\boxed{
\text{Linear output error}
\rightarrow
\text{ReLU}
\rightarrow
\text{hidden activation error}
}
$$

だけを扱います。

前の Notebook 01 で確認済みの、

$$
\Delta Z
=
\widetilde Z-Z
=
X\Delta W^{\mathsf T}
$$

および、

$$
\|\Delta Z\|_F
\le
\|X\|_F\|\Delta W\|_2
$$

を前提として使います。

今回の中心目標は、

$$
\boxed{
\|\widetilde H-H\|_F
\le
\|\widetilde Z-Z\|_F
}
$$

を理論と PyTorch の数値実験の両方で確認することです。

さらに、

$$
\boxed{
\|\widetilde H-H\|_F
\le
\|\widetilde Z-Z\|_F
\le
\|X\|_F\|\Delta W\|_2
}
$$

まで接続します。

## 今回やらないこと

- TT-SVD / TT-matrix の再実装
- 第2 Linear layer
- downstream operator norm
- logits error
- classification margin
- prediction
- accuracy
- Fashion-MNIST
- fine-tuning
- optimizer
- backward


## 1. 実行環境

今回は理論上の不等式を数値検証することが目的なので、CPU + `float64` を使います。

`float64` を使う理由：

- 丸め誤差を小さくする
- 理論上の等式・不等式を検証するときの余裕を確保する
- 実運用では `float32` が一般的でも、今回は correctness の確認が主目的

### このセルで初めて出る変数

- `dtype`
  - 意味：Notebook 全体で使う浮動小数点型
  - 数式対応：直接の数学記号はなし
  - 値：`torch.float64`
  - 用途：丸め誤差を抑えた数値検証

- `device`
  - 意味：Tensor を配置する計算デバイス
  - 値：CPU
  - 用途：再現性の高い小規模実験

- `rtol`, `atol`
  - 意味：`assert_close` で使う相対・絶対許容誤差
  - 用途：浮動小数点演算の差を許容しつつ等価性を検証

- `inequality_atol`
  - 意味：不等式判定にだけ加える小さい許容値
  - 用途：丸め誤差で境界をわずかに超えて見えることを避ける


In [19]:
import torch
from torch.testing import assert_close
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch.manual_seed(0)

dtype = torch.float64
device = torch.device("cpu")

rtol = 1e-12
atol = 1e-12
inequality_atol = 1e-12

print("dtype :", dtype)
print("device:", device)


dtype : torch.float64
device: cpu


## 2. ReLU の scalar 定義

ReLU（Rectified Linear Unit）は、実数 $x$ に対して、

$$
\operatorname{ReLU}(x)
=
\max(0,x)
$$

と定義されます。

場合分けすると、

$$
\operatorname{ReLU}(x)
=
\begin{cases}
x, & x\ge0,\\
0, & x<0.
\end{cases}
$$

つまり、

- 非負の値はそのまま通す
- 負の値は 0 にする

という関数です。


## 3. ReLU が scalar に対して 1-Lipschitz であること

示したいのは、任意の $a,b\in\mathbb R$ に対して、

$$
\boxed{
|\operatorname{ReLU}(a)-\operatorname{ReLU}(b)|
\le
|a-b|
}
$$

が成り立つことです。

### Case 1: $a\ge0,\ b\ge0$

$$
\operatorname{ReLU}(a)=a,
\qquad
\operatorname{ReLU}(b)=b.
$$

したがって、

$$
|\operatorname{ReLU}(a)-\operatorname{ReLU}(b)|
=
|a-b|.
$$

この場合は **等号成立**です。ReLU は誤差を縮めませんが、増幅もしません。

---

### Case 2: $a<0,\ b<0$

$$
\operatorname{ReLU}(a)=0,
\qquad
\operatorname{ReLU}(b)=0.
$$

よって、

$$
|\operatorname{ReLU}(a)-\operatorname{ReLU}(b)|
=
0
\le
|a-b|.
$$

この場合、ReLU 後の誤差は完全に消えます。

---

### Case 3: $a\ge0,\ b<0$

$$
\operatorname{ReLU}(a)=a,
\qquad
\operatorname{ReLU}(b)=0.
$$

したがって、

$$
|\operatorname{ReLU}(a)-\operatorname{ReLU}(b)|
=
a.
$$

一方、

$$
|a-b|
=
a-b.
$$

$b<0$ なので $-b>0$ であり、

$$
a-b>a.
$$

よって、

$$
|\operatorname{ReLU}(a)-\operatorname{ReLU}(b)|
<
|a-b|.
$$

---

### Case 4: $a<0,\ b\ge0$

$$
\operatorname{ReLU}(a)=0,
\qquad
\operatorname{ReLU}(b)=b.
$$

したがって、

$$
|\operatorname{ReLU}(a)-\operatorname{ReLU}(b)|
=
b.
$$

一方、

$$
|a-b|
=
b-a.
$$

$a<0$ なので $-a>0$ であり、

$$
b-a>b.
$$

よって、

$$
|\operatorname{ReLU}(a)-\operatorname{ReLU}(b)|
<
|a-b|.
$$

---

4ケースを合わせると、

$$
\boxed{
|\operatorname{ReLU}(a)-\operatorname{ReLU}(b)|
\le
|a-b|
}
$$

です。


## 4. 1-Lipschitz の意味

一般に関数 $f$ が $L$-Lipschitz とは、

$$
|f(a)-f(b)|
\le
L|a-b|
$$

を満たすことです。

ReLU では $L=1$ なので、

$$
|\operatorname{ReLU}(a)-\operatorname{ReLU}(b)|
\le
1\cdot|a-b|
$$

です。

重要なのは、

> ReLU は誤差を必ず縮める

という意味ではないことです。

両方の入力が非負なら、

$$
|\operatorname{ReLU}(a)-\operatorname{ReLU}(b)|
=
|a-b|
$$

なので、誤差はそのまま残ります。

したがって ReLU の 1-Lipschitz 性は、

> **誤差を増幅しない**

という意味です。


## 5. 行列・batch tensor への拡張

dense Linear の ReLU 前の出力を、

$$
Z\in\mathbb R^{B\times h}
$$

とします。

TT-rank truncation 後の近似 weight を使った ReLU 前の出力を、

$$
\widetilde Z\in\mathbb R^{B\times h}
$$

とします。

ここで、

- $B$: batch size
- $h$: hidden dimension
- 行軸 $i$: batch sample
- 列軸 $j$: hidden unit

です。

ReLU 後を、

$$
H_{ij}
=
\operatorname{ReLU}(Z_{ij}),
$$

$$
\widetilde H_{ij}
=
\operatorname{ReLU}(\widetilde Z_{ij})
$$

とします。

scalar の 1-Lipschitz 不等式に、

$$
a=\widetilde Z_{ij},
\qquad
b=Z_{ij}
$$

を代入すると、

$$
\boxed{
|\widetilde H_{ij}-H_{ij}|
\le
|\widetilde Z_{ij}-Z_{ij}|
}
$$

が各要素について成り立ちます。

つまり、

$$
|\Delta H_{ij}|
\le
|\Delta Z_{ij}|
$$

です。


## 6. Frobenius norm への拡張

要素ごとの不等式、

$$
|\widetilde H_{ij}-H_{ij}|
\le
|\widetilde Z_{ij}-Z_{ij}|
$$

の両辺は非負なので、二乗して、

$$
(\widetilde H_{ij}-H_{ij})^2
\le
(\widetilde Z_{ij}-Z_{ij})^2
$$

とできます。

これを全要素について足すと、

$$
\sum_{i=1}^{B}\sum_{j=1}^{h}
(\widetilde H_{ij}-H_{ij})^2
\le
\sum_{i=1}^{B}\sum_{j=1}^{h}
(\widetilde Z_{ij}-Z_{ij})^2.
$$

Frobenius norm は、

$$
\|A\|_F
=
\sqrt{
\sum_i\sum_j A_{ij}^2
}
$$

なので、

$$
\|\widetilde H-H\|_F^2
\le
\|\widetilde Z-Z\|_F^2.
$$

両辺は非負で、平方根は非負領域で単調増加なので、

$$
\boxed{
\|\widetilde H-H\|_F
\le
\|\widetilde Z-Z\|_F
}
$$

を得ます。

これは、

$$
\boxed{
\|\Delta H\|_F
\le
\|\Delta Z\|_F
}
$$

と同じです。


## 7. 実験1 — 符号パターンを含む小さな具体例

まず random tensor ではなく、各要素を手で追える例を使います。

### このセルで初めて出る変数

#### `z`

- 数式対応：
  $$
  Z
  $$
- 意味：dense Linear 側の **ReLU 前の出力（pre-activation）**
- shape：`(2, 4)`
- 軸0：サンプル
- 軸1：hidden unit
- 由来：ここでは理論確認用に手で与える
- 用途：ReLU 前の基準値

#### `z_tilde`

- 数式対応：
  $$
  \widetilde Z
  $$
- 意味：近似 weight 側の ReLU 前出力
- shape：`(2, 4)`
- 軸の意味：`z` と同じ
- 由来：ここでは手で与える
- 用途：`z` との差が ReLU でどう変化するかを見る

#### `h`

- 数式対応：
  $$
  H=\operatorname{ReLU}(Z)
  $$
- 意味：dense 側 hidden activation
- shape：`(2, 4)`
- 由来：`torch.relu(z)`

#### `h_tilde`

- 数式対応：
  $$
  \widetilde H=\operatorname{ReLU}(\widetilde Z)
  $$
- 意味：近似 weight 側 hidden activation
- shape：`(2, 4)`
- 由来：`torch.relu(z_tilde)`

#### `delta_z`

- 数式対応：
  $$
  \Delta Z=\widetilde Z-Z
  $$
- 意味：ReLU 前の Linear output error
- shape：`(2, 4)`
- 由来：`z_tilde - z`
- 用途：ReLU 前の誤差

#### `delta_h`

- 数式対応：
  $$
  \Delta H=\widetilde H-H
  $$
- 意味：ReLU 後の hidden activation error
- shape：`(2, 4)`
- 由来：`h_tilde - h`
- 用途：`delta_z` と比較して ReLU が誤差を増幅しないことを確認

#### `elementwise_holds`

- 意味：各要素で
  $$
  |\Delta H_{ij}|
  \le
  |\Delta Z_{ij}|
  $$
  が成立したかを持つ boolean tensor
- shape：`(2, 4)`
- 用途：要素ごとの 1-Lipschitz 性の数値確認


In [20]:
# 手で追える toy pre-activation
z = torch.tensor(
    [
        [2.0, -2.0, 3.0, -4.0],
        [1.0, -5.0, -1.0, 4.0],
    ],
    dtype=dtype,
    device=device,
)

z_tilde = torch.tensor(
    [
        [5.0, -7.0, -2.0, 2.0],
        [3.0, -1.0, 2.0, 1.0],
    ],
    dtype=dtype,
    device=device,
)

# TODO 1:
# 1. h, h_tilde を ReLU で計算
h = torch.relu(z)
h_tilde = torch.relu(z_tilde)

# 2. delta_z, delta_h を計算
delta_z = z_tilde - z
delta_h = h_tilde - h

# 3. |delta_h| <= |delta_z| を要素ごとに確認
assert torch.all(delta_h.abs() <= delta_z.abs() + inequality_atol)

# 4. 関連する tensor の shape がすべて (2, 4) であることを確認
assert z.shape == (2, 4)
assert z_tilde.shape == (2, 4)
assert h.shape == (2, 4)
assert h_tilde.shape == (2, 4)
assert delta_z.shape == (2, 4)
assert delta_h.shape == (2, 4)


## 8. 各要素がどの符号ケースか確認する

同じ `z` と `z_tilde` を使い、各位置が scalar 証明のどのケースに対応するかを表示します。

### このセルで初めて出る変数

#### `case_name`

- 意味：各要素の $(Z_{ij},\widetilde Z_{ij})$ の符号パターン名
- shape：scalar string
- 由来：2つの値の符号判定
- 用途：理論の4ケースと数値結果を対応付ける

#### `abs_delta_z_ij`

- 数式対応：
  $$
  |\Delta Z_{ij}|
  $$
- 意味：1要素の ReLU 前誤差の絶対値
- 用途：`abs_delta_h_ij` と比較

#### `abs_delta_h_ij`

- 数式対応：
  $$
  |\Delta H_{ij}|
  $$
- 意味：1要素の ReLU 後誤差の絶対値
- 用途：ReLU により誤差が保存・消失・縮小したか確認


In [21]:
def sign_case(z_value: float, z_tilde_value: float) -> str:
    """Z と Z_tilde の符号から ReLU のケース名を返す。"""
    if z_value >= 0.0 and z_tilde_value >= 0.0:
        return "both nonnegative"
    if z_value < 0.0 and z_tilde_value < 0.0:
        return "both negative"
    if z_value >= 0.0 and z_tilde_value < 0.0:
        return "dense nonnegative, approximate negative"
    return "dense negative, approximate nonnegative"


for i in range(z.shape[0]):
    for j in range(z.shape[1]):
        z_value = z[i, j].item()
        z_tilde_value = z_tilde[i, j].item()

        case_name = sign_case(
            z_value=z_value,
            z_tilde_value=z_tilde_value,
        )

        abs_delta_z_ij = delta_z[i, j].abs().item()
        abs_delta_h_ij = delta_h[i, j].abs().item()

        print(
            f"({i}, {j}): "
            f"Z={z_value:.1f}, Z_tilde={z_tilde_value:.1f}"
        )
        print(f"  case       : {case_name}")
        print(f"  |delta_H|  : {abs_delta_h_ij:.1f}")
        print(f"  |delta_Z|  : {abs_delta_z_ij:.1f}")
        print(f"  inequality : {abs_delta_h_ij <= abs_delta_z_ij}")
        print()


(0, 0): Z=2.0, Z_tilde=5.0
  case       : both nonnegative
  |delta_H|  : 3.0
  |delta_Z|  : 3.0
  inequality : True

(0, 1): Z=-2.0, Z_tilde=-7.0
  case       : both negative
  |delta_H|  : 0.0
  |delta_Z|  : 5.0
  inequality : True

(0, 2): Z=3.0, Z_tilde=-2.0
  case       : dense nonnegative, approximate negative
  |delta_H|  : 3.0
  |delta_Z|  : 5.0
  inequality : True

(0, 3): Z=-4.0, Z_tilde=2.0
  case       : dense negative, approximate nonnegative
  |delta_H|  : 2.0
  |delta_Z|  : 6.0
  inequality : True

(1, 0): Z=1.0, Z_tilde=3.0
  case       : both nonnegative
  |delta_H|  : 2.0
  |delta_Z|  : 2.0
  inequality : True

(1, 1): Z=-5.0, Z_tilde=-1.0
  case       : both negative
  |delta_H|  : 0.0
  |delta_Z|  : 4.0
  inequality : True

(1, 2): Z=-1.0, Z_tilde=2.0
  case       : dense negative, approximate nonnegative
  |delta_H|  : 2.0
  |delta_Z|  : 3.0
  inequality : True

(1, 3): Z=4.0, Z_tilde=1.0
  case       : both nonnegative
  |delta_H|  : 3.0
  |delta_Z|  : 3.0
  inequ

## 9. 実験2 — Frobenius norm で全体誤差を比較する

要素ごとの不等式が全要素で成り立つので、Frobenius norm でも、

$$
\|\Delta H\|_F
\le
\|\Delta Z\|_F
$$

となるはずです。

### このセルで初めて出る変数

#### `delta_z_fro`

- 数式対応：
  $$
  \|\Delta Z\|_F
  $$
- 意味：ReLU 前の誤差を全要素で集約した Frobenius norm
- shape：0次元 scalar tensor
- 由来：`delta_z`
- 用途：ReLU 前の全体誤差

#### `delta_h_fro`

- 数式対応：
  $$
  \|\Delta H\|_F
  $$
- 意味：ReLU 後の hidden activation error の Frobenius norm
- shape：0次元 scalar tensor
- 由来：`delta_h`
- 用途：`delta_z_fro` と比較して non-expansive 性を確認


In [22]:
# TODO 2:
# delta_z と delta_h の Frobenius norm を計算し、
# ||Delta_H||_F <= ||Delta_Z||_F を確認してください。
delta_z_fro = torch.linalg.matrix_norm(delta_z, ord="fro")
delta_h_fro = torch.linalg.matrix_norm(delta_h, ord="fro")

assert float(delta_h_fro) <= float(delta_z_fro) + inequality_atol, (
    "not ||Delta_H||_F <= ||Delta_Z||_F"
)


## 10. 実験3 — Linear → ReLU の誤差伝播

次は、手で与えた $Z,\widetilde Z$ ではなく、Linear の式から作ります。

今回は TT decomposition 自体は再実装しません。

`w_tilde` は、

> **TT-rank truncation 後に dense へ復元された近似 weight**

を模した toy weight として、

$$
\widetilde W=W+\text{small perturbation}
$$

で作ります。

### shape

$$
X\in\mathbb R^{B\times d_{\mathrm{in}}},
$$

$$
W,\widetilde W,\Delta W
\in
\mathbb R^{h\times d_{\mathrm{in}}},
$$

$$
b\in\mathbb R^h,
$$

$$
Z,\widetilde Z,\Delta Z,H,\widetilde H,\Delta H
\in
\mathbb R^{B\times h}.
$$

今回は、

$$
B=4,
\qquad
d_{\mathrm{in}}=6,
\qquad
h=5.
$$

### このセルで初めて出る変数

#### `batch_size`

- 意味：入力サンプル数 $B$
- 値：4
- 用途：`x` の行数を決める

#### `in_features`

- 数式対応：$d_{\mathrm{in}}$
- 意味：1サンプル当たりの入力特徴数
- 値：6
- 用途：`x` の列数と weight の入力軸

#### `hidden_features`

- 数式対応：$h$
- 意味：hidden unit 数
- 値：5
- 用途：weight の出力軸と $Z,H$ の列数

#### `x`

- 数式対応：$X$
- shape：`(B, d_in) = (4, 6)`
- 軸0：サンプル
- 軸1：入力特徴
- 由来：固定 seed の random tensor
- 用途：dense / approximate の両方へ同じ入力を与える

#### `w`

- 数式対応：$W$
- shape：`(h, d_in) = (5, 6)`
- 軸0：hidden unit
- 軸1：入力特徴
- 由来：random tensor
- 用途：基準となる dense weight

#### `small_perturbation`

- 意味：TT-rank truncation による weight approximation error を模した小さい摂動
- shape：`(5, 6)`
- 由来：random tensor を小さい係数でスケール
- 用途：`w_tilde` を作る

#### `w_tilde`

- 数式対応：$\widetilde W$
- shape：`(5, 6)`
- 由来：`w + small_perturbation`
- 用途：truncation 後の近似 weight の toy 代理

#### `b`

- 数式対応：$b$
- shape：`(h,) = (5,)`
- 軸0：hidden unit
- 由来：random tensor
- 用途：両 Linear で共通の bias


In [23]:
batch_size = 4
in_features = 6
hidden_features = 5

x = torch.randn(
    batch_size,
    in_features,
    dtype=dtype,
    device=device,
)

w = torch.randn(
    hidden_features,
    in_features,
    dtype=dtype,
    device=device,
)

small_perturbation = 0.05 * torch.randn(
    hidden_features,
    in_features,
    dtype=dtype,
    device=device,
)

w_tilde = w + small_perturbation

b = torch.randn(
    hidden_features,
    dtype=dtype,
    device=device,
)

assert x.shape == (batch_size, in_features)
assert w.shape == (hidden_features, in_features)
assert w_tilde.shape == w.shape
assert b.shape == (hidden_features,)

print("x.shape       =", tuple(x.shape))
print("w.shape       =", tuple(w.shape))
print("w_tilde.shape =", tuple(w_tilde.shape))
print("b.shape       =", tuple(b.shape))


x.shape       = (4, 6)
w.shape       = (5, 6)
w_tilde.shape = (5, 6)
b.shape       = (5,)


## 11. $Z,\widetilde Z,\Delta W,\Delta Z$ を作る

### このセルで初めて出る変数

#### `z`

- 数式対応：
  $$
  Z=XW^{\mathsf T}+b
  $$
- 意味：dense Linear の ReLU 前出力
- shape：`(B, h) = (4, 5)`
- 軸0：サンプル
- 軸1：hidden unit
- 由来：`x @ w.T + b`
- 用途：dense 側 pre-activation

#### `z_tilde`

- 数式対応：
  $$
  \widetilde Z=X\widetilde W^{\mathsf T}+b
  $$
- 意味：近似 weight 側の ReLU 前出力
- shape：`(4, 5)`
- 由来：`x @ w_tilde.T + b`
- 用途：`z` と比較

#### `delta_w`

- 数式対応：
  $$
  \Delta W=\widetilde W-W
  $$
- 意味：weight approximation error
- shape：`(h, d_in) = (5, 6)`
- 由来：`w_tilde - w`
- 用途：前の Notebook 01 との接続

#### `delta_z`

- 数式対応：
  $$
  \Delta Z=\widetilde Z-Z
  $$
- 意味：Linear output error / pre-activation error
- shape：`(B, h) = (4, 5)`
- 由来：`z_tilde - z`
- 用途：ReLU 前の誤差

#### `delta_z_from_formula`

- 数式対応：
  $$
  X\Delta W^{\mathsf T}
  $$
- 意味：Notebook 01 で導出した式から直接計算した $\Delta Z$
- shape：`(4, 5)`
- 由来：`x @ delta_w.T`
- 用途：
  $$
  \Delta Z=X\Delta W^{\mathsf T}
  $$
  を数値検証する


In [24]:
# TODO 3:
# x, w, w_tilde, b から z, z_tilde を計算し、
# delta_w, delta_z を求めてください。
z = x@w.T + b
z_tilde = x @ w_tilde.T + b
delta_w = w_tilde - w
delta_z = z_tilde - z

# さらに Notebook 01 の
# Delta_Z = X Delta_W^T
delta_z_from_formula = x @ delta_w.T
# を別計算して、両者が一致することを確認してください。
assert_close(delta_z, delta_z_from_formula, rtol=rtol, atol=atol)


## 12. ReLU を通して hidden activation error を作る

### このセルで初めて出る変数

#### `h`

- 数式対応：
  $$
  H=\operatorname{ReLU}(Z)
  $$
- 意味：dense 側 hidden activation
- shape：`(B, h) = (4, 5)`
- 由来：`torch.relu(z)`

#### `h_tilde`

- 数式対応：
  $$
  \widetilde H=\operatorname{ReLU}(\widetilde Z)
  $$
- 意味：近似 weight 側 hidden activation
- shape：`(4, 5)`
- 由来：`torch.relu(z_tilde)`

#### `delta_h`

- 数式対応：
  $$
  \Delta H=\widetilde H-H
  $$
- 意味：hidden activation error
- shape：`(4, 5)`
- 由来：`h_tilde - h`
- 用途：
  $$
  \|\Delta H\|_F
  \le
  \|\Delta Z\|_F
  $$
  を検証する


In [25]:
# TODO 4:
# z, z_tilde に ReLU を適用して h, h_tilde を作り、
# delta_h を計算してください。
h = torch.relu(z)
h_tilde = torch.relu(z_tilde)
delta_h = h_tilde - h

# 各要素で |Delta_H_ij| <= |Delta_Z_ij| が成り立つことを確認してください。
delta_z = z_tilde - z
assert torch.all(delta_h.abs() <= delta_z.abs() + inequality_atol)


## 13. Norm と Linear output error bound を計算する

ここで Notebook 01 の結果と接続します。

### このセルで初めて出る変数

#### `x_fro`

- 数式対応：
  $$
  \|X\|_F
  $$
- 意味：入力 batch 全体の Frobenius norm
- shape：scalar tensor
- 由来：`x`
- 用途：Linear output error の上界を作る

#### `delta_w_spectral`

- 数式対応：
  $$
  \|\Delta W\|_2
  $$
- 意味：weight error の matrix spectral norm
- shape：scalar tensor
- 由来：`delta_w`
- 用途：入力方向に対する weight error の最大増幅率

#### `delta_z_fro`

- 数式対応：
  $$
  \|\Delta Z\|_F
  $$
- 意味：実際に観測された pre-activation error
- shape：scalar tensor
- 由来：`delta_z`

#### `delta_h_fro`

- 数式対応：
  $$
  \|\Delta H\|_F
  $$
- 意味：実際に観測された hidden activation error
- shape：scalar tensor
- 由来：`delta_h`

#### `linear_bound`

- 数式対応：
  $$
  \|X\|_F\|\Delta W\|_2
  $$
- 意味：Notebook 01 で確認した Linear output error の理論上界
- shape：scalar tensor
- 由来：`x_fro * delta_w_spectral`
- 用途：
  $$
  \|\Delta H\|_F
  \le
  \|\Delta Z\|_F
  \le
  \|X\|_F\|\Delta W\|_2
  $$
  を検証する


In [28]:
# TODO 5:
# ||X||_F, ||Delta_W||_2, ||Delta_Z||_F, ||Delta_H||_F を計算し、
#
# ||Delta_H||_F
# <= ||Delta_Z||_F
# <= ||X||_F ||Delta_W||_2
#
# を数値的に確認してください。
# X の Frobenius norm: ||X||_F
x_fro = torch.linalg.matrix_norm(x, ord="fro")

# Delta_W の spectral norm: ||Delta_W||_2 = sigma_max(Delta_W)
delta_w_spectral = torch.linalg.matrix_norm(delta_w, ord=2)

# Delta_Z の Frobenius norm: ||Delta_Z||_F
delta_z_fro = torch.linalg.matrix_norm(delta_z, ord="fro")

# Delta_H の Frobenius norm: ||Delta_H||_F
delta_h_fro = torch.linalg.matrix_norm(delta_h, ord="fro")

# Linear output error bound: ||X||_F ||Delta_W||_2
linear_bound = x_fro * delta_w_spectral
linear_bound = x_fro * delta_w_spectral

assert float(delta_h_fro) <= float(delta_z_fro) + inequality_atol
assert float(delta_z_fro) <= float(linear_bound) + inequality_atol
# ReLU による non-expansive 性
assert delta_h_fro <= delta_z_fro + inequality_atol

# Notebook 01 からの Linear output error bound
assert delta_z_fro <= linear_bound + inequality_atol

# 2つを結合した最終的な bound
assert delta_h_fro <= linear_bound + inequality_atol

print(f"||Delta_H||_F             = {delta_h_fro.item():.12f}")
print(f"||Delta_Z||_F             = {delta_z_fro.item():.12f}")
print(f"||X||_F                   = {x_fro.item():.12f}")
print(f"||Delta_W||_2             = {delta_w_spectral.item():.12f}")
print(f"||X||_F ||Delta_W||_2     = {linear_bound.item():.12f}")


||Delta_H||_F             = 0.337250011229
||Delta_Z||_F             = 0.519605165542
||X||_F                   = 5.091422875106
||Delta_W||_2             = 0.175429273041
||X||_F ||Delta_W||_2     = 0.893184613722


## 14. 補足 — もう一つの Linear output error bound

Notebook 01 で扱ったもう一つの上界も確認します。

$$
\|\Delta Z\|_F
\le
\|X\|_2\|\Delta W\|_F.
$$

ReLU の 1-Lipschitz 性と合わせると、

$$
\boxed{
\|\Delta H\|_F
\le
\|\Delta Z\|_F
\le
\|X\|_2\|\Delta W\|_F
}
$$

です。

### このセルで初めて出る変数

#### `x_spectral`

- 数式対応：
  $$
  \|X\|_2
  $$
- 意味：入力行列 $X$ の spectral norm
- shape：scalar tensor
- 由来：`x`
- 用途：別形式の Linear error bound

#### `delta_w_fro`

- 数式対応：
  $$
  \|\Delta W\|_F
  $$
- 意味：weight approximation error 全体の Frobenius norm
- shape：scalar tensor
- 由来：`delta_w`

#### `linear_bound_alt`

- 数式対応：
  $$
  \|X\|_2\|\Delta W\|_F
  $$
- 意味：別形式の Linear output error 上界
- shape：scalar tensor
- 用途：`delta_z_fro` と `delta_h_fro` の上界になっているか確認


In [ ]:
# TODO 6（補足）:
# ||X||_2 と ||Delta_W||_F を使う別の上界
#
# ||Delta_H||_F
# <= ||Delta_Z||_F
# <= ||X||_2 ||Delta_W||_F
#
# も確認してください。
# X の spectral norm: ||X||_2
x_spectral = torch.linalg.matrix_norm(x, ord=2)

# Delta_W の Frobenius norm: ||Delta_W||_F
delta_w_fro = torch.linalg.matrix_norm(delta_w, ord="fro")

# 別形式の Linear output error bound
linear_bound_alt = x_spectral * delta_w_fro

assert delta_h_fro <= delta_z_fro + inequality_atol
assert delta_z_fro <= linear_bound_alt + inequality_atol
assert delta_h_fro <= linear_bound_alt + inequality_atol

print(f"||X||_2                   = {x_spectral.item():.12f}")
print(f"||Delta_W||_F             = {delta_w_fro.item():.12f}")
print(f"||X||_2 ||Delta_W||_F     = {linear_bound_alt.item():.12f}")

assert float(delta_h_fro) <= float(delta_z_fro) + inequality_atol
assert float(delta_z_fro) <= float(linear_bound_alt) + inequality_atol


## 15. まとめ

今回確認した内容は次の通りです。

### 1. ReLU は scalar に対して 1-Lipschitz

$$
\boxed{
|\operatorname{ReLU}(a)-\operatorname{ReLU}(b)|
\le
|a-b|
}
$$

です。

### 2. 要素ごとの hidden activation error は pre-activation error を超えない

$$
\boxed{
|\Delta H_{ij}|
\le
|\Delta Z_{ij}|
}
$$

です。

### 3. Frobenius norm でも誤差は増幅されない

$$
\boxed{
\|\Delta H\|_F
\le
\|\Delta Z\|_F
}
$$

です。

### 4. Linear output error bound と接続できる

前の Notebook 01 で確認した、

$$
\|\Delta Z\|_F
\le
\|X\|_F\|\Delta W\|_2
$$

と合わせると、

$$
\boxed{
\|\Delta H\|_F
\le
\|\Delta Z\|_F
\le
\|X\|_F\|\Delta W\|_2
}
$$

です。

また、

$$
\boxed{
\|\Delta H\|_F
\le
\|\Delta Z\|_F
\le
\|X\|_2\|\Delta W\|_F
}
$$

も確認しました。

### 5. ReLU は誤差を必ず縮めるわけではない

両方の pre-activation が非負なら、

$$
|\Delta H_{ij}|
=
|\Delta Z_{ij}|
$$

となり、誤差は保存されます。

両方が負なら ReLU 後の誤差は 0 になり、符号が異なる場合は誤差が縮みます。

---

今回の到達点は、

$$
\boxed{
\text{Linear output error}
\rightarrow
\text{ReLU}
\rightarrow
\text{hidden activation error}
}
$$

までです。

次の第2 Linear layer、logits error、classification margin、accuracy、fine-tuning には進んでいません。
